# 🎬 Fandango Ratings Capstone Project
## Overview

If you are planning on going out to see a movie, how well can you trust online reviews and ratings? *Especially* if the same company showing the rating *also* makes money by selling movie tickets. Do they have a bias towards rating movies higher than they should be rated?

### Goal
**Determine if Fandango's ratings in 2015 were biased towards rating movies higher to sell more tickets**, using data from the [FiveThirtyEight article: *Be Suspicious Of Online Movie Ratings, Especially Fandango's*](http://fivethirtyeight.com/features/fandango-movies-ratings/).

---

### The Data

**`fandango_scrape.csv`** — Every film 538 pulled from Fandango.

| Column | Definition |
|--------|-----------|
| FILM | The movie |
| STARS | Number of stars displayed on Fandango.com |
| RATING | The actual average score pulled from the HTML |
| VOTES | Number of people who reviewed the film |

**`all_sites_scores.csv`** — Every film with ratings from RT, Metacritic, and IMDB.

| Column | Definition |
|--------|-----------|
| FILM | The film |
| RottenTomatoes | RT Tomatometer score |
| RottenTomatoes_User | RT user score |
| Metacritic | Metacritic critic score |
| Metacritic_User | Metacritic user score |
| IMDB | IMDb user score |
| Metacritic_user_vote_count | Number of Metacritic user votes |
| IMDB_user_vote_count | Number of IMDB user votes |


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## 2. Exploring Fandango Ratings

Let's first explore the Fandango ratings to see if our analysis agrees with the article's conclusion.

In [ ]:
fandango = pd.read_csv("fandango_scrape.csv")

### 2.1 DataFrame Overview

In [ ]:
fandango.head(10)

In [ ]:
fandango.info()

In [ ]:
fandango.describe()

### 2.2 Popularity vs Rating

Is there a relationship between how popular a movie is (votes) and its rating?

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(y='VOTES', x='RATING', data=fandango)
plt.title('Fandango: Votes vs Rating')
plt.show()

### 2.3 Correlation Matrix

In [ ]:
# Select only numeric columns and compute correlation
fandango_numeric = fandango.select_dtypes(include='number')
fandango_numeric.corr()

### 2.4 Extract Year from Film Title

Each film title follows the format: `Film Title Name (Year)`.  
We use regex to extract the 4-digit year inside the parentheses.

In [ ]:
fandango['YEAR'] = fandango['FILM'].str.extract(r'\((\d{4})\)')
fandango.head()

### 2.5 Movies Per Year

In [ ]:
fandango['YEAR'].value_counts()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x='YEAR', data=fandango, hue='YEAR', legend=False)
plt.title('Number of Movies per Year')
plt.show()

### 2.6 Top 10 Most Voted Movies

In [ ]:
fandango.sort_values('VOTES', ascending=False).head(10)

### 2.7 Remove Unreviewed Films

In [ ]:
# How many movies have zero votes?
print(f"Movies with zero votes: {fandango[fandango['VOTES'] == 0].shape[0]}")

# Keep only reviewed films
fandango_reviewed = fandango[fandango['VOTES'] != 0].copy()
print(f"Reviewed films: {len(fandango_reviewed)}")

### 2.8 STARS vs True RATING Distribution

Due to HTML and star rating displays, the true user rating may differ slightly from the displayed rating.  
The KDE curves are clipped to (0, 5) to stay within the valid rating range.

> **Note:** KDE smoothing may cause slight spillover beyond the actual min/max values — this is expected behavior, not a data error.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=fandango_reviewed, x='RATING', label='True Rating', clip=(0, 5), fill=True)
sns.kdeplot(data=fandango_reviewed, x='STARS', label='Stars Displayed', clip=(0, 5), fill=True)
plt.xlabel('Rating')
plt.title('Fandango STARS vs True RATING Distribution')
plt.legend()
plt.show()

### 2.9 Quantifying the STARS vs RATING Difference

We create a new column `STARS_DIFF` = STARS − RATING, rounded to 1 decimal place.

In [ ]:
fandango_reviewed['STARS_DIFF'] = (fandango_reviewed['STARS'] - fandango_reviewed['RATING']).round(1)
fandango_reviewed.head()

In [ ]:
plt.figure(figsize=(14, 6))
ax = plt.gca()
ax.tick_params(axis='both', labelsize=11)
sns.countplot(x='STARS_DIFF', data=fandango_reviewed, hue='STARS_DIFF', palette='magma', legend=False)
plt.title('Count of STARS vs RATING Differences')
plt.xlabel('STARS - RATING')
plt.show()

**Which movie had the largest (1 star) difference between displayed and true rating?**

In [ ]:
fandango_reviewed[fandango_reviewed['STARS_DIFF'] == 1]

---
## 3. Comparing Fandango to Other Sites

Let's compare Fandango's ratings to those from Rotten Tomatoes, Metacritic, and IMDB.

In [ ]:
all_sites = pd.read_csv("all_sites_scores.csv")

In [ ]:
all_sites.head()

In [ ]:
all_sites.info()

### 3.1 Rotten Tomatoes: Critics vs Users

Is there agreement between RT critics and RT users?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='RottenTomatoes', y='RottenTomatoes_User', data=all_sites,
                color='tomato', edgecolor='white', s=80, alpha=0.7)
plt.title('RT Critics vs RT User Ratings')
plt.xlabel('Critics Score')
plt.ylabel('User Score')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

### 3.2 RT Critic vs User Score Difference

`RT_Diff` = Critics Rating − User Rating  
- Values close to **0** → critics and users agree  
- **Positive** values → critics rated higher than users  
- **Negative** values → users rated higher than critics

In [ ]:
all_sites['RT_Diff'] = (all_sites['RottenTomatoes'] - all_sites['RottenTomatoes_User']).round(1)

# Mean Absolute Difference
print(f"Mean Absolute Difference: {all_sites['RT_Diff'].abs().mean().round(2)}")

In [ ]:
plt.figure(figsize=(15, 6))
sns.histplot(x='RT_Diff', data=all_sites, kde=True, bins=25)
plt.title('Distribution of RT Critics vs User Score Difference')
plt.xlabel('RT Diff (Critics - Users)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(x=all_sites['RT_Diff'].abs(), kde=True, bins=25)
plt.title('Absolute Difference Between RT Critics and User Score')
plt.xlabel('Absolute RT Diff')
plt.show()

### 3.3 Movies with Largest Rating Gaps

In [ ]:
# Top 5 movies users rated higher than critics
print("Users rated MUCH higher than critics:")
all_sites[['FILM', 'RT_Diff']].sort_values('RT_Diff', ascending=True).head(5)

In [ ]:
# Top 5 movies critics rated higher than users
print("Critics rated MUCH higher than users:")
all_sites[['FILM', 'RT_Diff']].sort_values('RT_Diff', ascending=False).head(5)

### 3.4 Metacritic: Critics vs Users

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(y='Metacritic_User', x='Metacritic', data=all_sites,
                color='royalblue', edgecolor='white', s=80, alpha=0.7)
plt.xlim(0, 100)
plt.ylim(0, 10)
plt.title('Metacritic Critics vs User Score', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Critics Score', fontsize=12)
plt.ylabel('User Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

### 3.5 IMDB: Vote Count Comparison

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(y='IMDB_user_vote_count', x='Metacritic_user_vote_count', data=all_sites,
                color='goldenrod', edgecolor='white', s=80, alpha=0.7)
plt.title('Metacritic vs IMDB User Vote Counts', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Metacritic Vote Count', fontsize=12)
plt.ylabel('IMDB Vote Count', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

**Which movie has the highest IMDB user vote count?**

In [ ]:
all_sites[all_sites['IMDB_user_vote_count'] == all_sites['IMDB_user_vote_count'].max()]

**Which movie has the highest Metacritic user vote count?**

In [ ]:
all_sites.loc[all_sites['Metacritic_user_vote_count'].idxmax()]

---
## 4. Fandango vs All Sites

Now let's merge both DataFrames and normalize all ratings to a 0–5 scale for a fair comparison.

In [ ]:
# Inner merge — only keep movies present in both DataFrames
merged_df = pd.merge(fandango, all_sites, how='inner', on='FILM')
print(f"Merged DataFrame shape: {merged_df.shape}")
merged_df.head()

### 4.1 Normalize Ratings to 0–5 Scale

Different platforms use different scales:
- Rotten Tomatoes & Metacritic → 0–100 → divide by **20**
- Metacritic User & IMDB → 0–10 → divide by **2**

In [ ]:
merged_df['RT_Norm']    = np.round(merged_df['RottenTomatoes'] / 20, 1)
merged_df['RTU_Norm']   = np.round(merged_df['RottenTomatoes_User'] / 20, 1)
merged_df['Meta_Norm']  = np.round(merged_df['Metacritic'] / 20, 1)
merged_df['Meta_U_Norm']= np.round(merged_df['Metacritic_User'] / 2, 1)
merged_df['IMDB_Norm']  = np.round(merged_df['IMDB'] / 2, 1)

merged_df.head()

### 4.2 Normalized Scores DataFrame

In [ ]:
norm_scores = merged_df[['STARS', 'RATING', 'RT_Norm', 'RTU_Norm',
                          'Meta_Norm', 'Meta_U_Norm', 'IMDB_Norm']].copy()
norm_scores.head()

### 4.3 Distribution of Normalized Ratings Across All Sites

Do Fandango's ratings stand out compared to other platforms?

In [ ]:
plt.figure(figsize=(13, 6))

for col in norm_scores.columns:
    sns.kdeplot(data=norm_scores, x=col, label=col, clip=(0, 5), fill=True, alpha=0.3)

plt.xlabel('Normalized Rating')
plt.title('Distribution of Normalized Ratings Across All Sites')
plt.legend()
plt.show()

**Observation:** Fandango's STARS distribution is heavily skewed to the right (clustering around 4.0–4.5), while other platforms show a wider, more balanced spread.

### 4.4 Fandango STARS vs RT Critics — Direct Comparison

In [ ]:
plt.figure(figsize=(15, 6))
sns.kdeplot(data=norm_scores, x='RT_Norm', label='RT Critic Rating', clip=(0, 5), fill=True, alpha=0.3)
sns.kdeplot(data=norm_scores, x='STARS', label='Fandango Stars', clip=(0, 5), fill=True, alpha=0.3)
plt.xlabel('Normalized Rating')
plt.title('Fandango STARS vs RT Critic Rating Distribution')
plt.legend()
plt.show()

### 4.5 Histogram of All Normalized Scores (Optional)

In [ ]:
plt.figure(figsize=(15, 6))
sns.histplot(data=norm_scores, bins=20, kde=False, alpha=0.5)
plt.xlabel('Normalized Rating')
plt.title('Histogram of Normalized Ratings Across All Sites')
plt.legend(labels=norm_scores.columns)
plt.show()

### 4.6 Clustermap of Normalized Scores

In [ ]:
g = sns.clustermap(data=norm_scores, col_cluster=False, figsize=(7, 7))
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=5)
plt.show()

---
## 5. The 10 Worst Rated Movies

Based on RT Critic ratings, what are the lowest rated movies — and how does Fandango rate them?

In [ ]:
# Add FILM column back for identification
norm_scores['FILM'] = merged_df['FILM']

# Top 10 worst movies by RT Critic score
worst_10 = norm_scores.sort_values('RT_Norm').head(10)
worst_10[['FILM', 'STARS', 'RATING', 'RT_Norm', 'RTU_Norm', 'Meta_Norm', 'Meta_U_Norm', 'IMDB_Norm']]

### 5.1 Distribution of Ratings for the 10 Worst Movies

In [ ]:
plt.figure(figsize=(15, 6))

for col in ['STARS', 'RATING', 'RT_Norm', 'RTU_Norm', 'Meta_Norm', 'Meta_U_Norm', 'IMDB_Norm']:
    sns.kdeplot(data=worst_10, x=col, label=col, clip=(0, 5), fill=True, alpha=0.3)

plt.xlabel('Normalized Rating')
plt.title("Ratings for RT Critics' 10 Worst Reviewed Films")
plt.legend()
plt.show()

---
## 6. Conclusion

![Taken 3](https://upload.wikimedia.org/wikipedia/en/6/6f/Taken_3_poster.jpg)

**Fandango is consistently displaying 3–4 star ratings for films that are clearly bad across every other platform.**

The biggest offender: **[Taken 3](https://www.youtube.com/watch?v=tJrfImRCHJ0)** — Fandango displayed **4.5 stars** for a film with an [average rating of just 1.86](https://en.wikipedia.org/wiki/Taken_3#Critical_response) across all other platforms.

### Key Findings
- Fandango's **displayed STARS** are consistently higher than the true underlying **RATING**
- Fandango ratings cluster heavily around **4.0–4.5**, unlike other platforms which show a broader distribution
- Even the **worst reviewed films** receive inflated scores on Fandango
- This bias likely exists because Fandango profits from ticket sales — higher ratings encourage more purchases
